In [6]:
import sys, torch
print(sys.executable)
print(torch.__version__)

c:\Users\rfern\Desktop\python-projects\ml-zoomcamp-2026\homework\08-deep-learning\.venv\Scripts\python.exe
2.9.0+cpu


In [7]:
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [8]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = datasets.ImageFolder("data/train", transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder("data/test", transform=evaluation_transform)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

print(train_dataset.class_to_idx)
print(len(train_dataset), len(evaluation_dataset))

{'curly': 0, 'straight': 1}
800 201


In [11]:
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)


model = HairModel().to(device)

In [13]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

print(sum(p.numel() for p in model.parameters()))

20073473


In [14]:
import json


def run_epochs(num_epochs):
    history = {"train_loss": [], "train_accuracy": [],
               "evaluation_loss": [], "evaluation_accuracy": []}

    for epoch in range(num_epochs):
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * images.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5).float() == labels).sum().item()
            total += images.size(0)

        history["train_loss"].append(loss_sum / total)
        history["train_accuracy"].append(correct / total)

        model.eval()
        loss_sum, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in evaluation_loader:
                images = images.to(device)
                labels = labels.float().unsqueeze(1).to(device)

                logits = model(images)
                loss = criterion(logits, labels)

                loss_sum += loss.item() * images.size(0)
                correct += ((torch.sigmoid(logits) >= 0.5).float() == labels).sum().item()
                total += images.size(0)

        history["evaluation_loss"].append(loss_sum / total)
        history["evaluation_accuracy"].append(correct / total)

        print(f"epoch {epoch + 1}: "
              f"train_loss={history['train_loss'][-1]:.4f} "
              f"train_acc={history['train_accuracy'][-1]:.4f} "
              f"eval_loss={history['evaluation_loss'][-1]:.4f} "
              f"eval_acc={history['evaluation_accuracy'][-1]:.4f}")

    return history

In [15]:
history_baseline = run_epochs(10)

with open("history_baseline.json", "w") as f:
    json.dump(history_baseline, f, indent=2)

epoch 1: train_loss=0.6385 train_acc=0.6450 eval_loss=0.6010 eval_acc=0.6567
epoch 2: train_loss=0.5441 train_acc=0.7175 eval_loss=0.6003 eval_acc=0.6517
epoch 3: train_loss=0.4955 train_acc=0.7550 eval_loss=0.6405 eval_acc=0.6219
epoch 4: train_loss=0.4330 train_acc=0.7913 eval_loss=0.6274 eval_acc=0.6915
epoch 5: train_loss=0.4048 train_acc=0.8087 eval_loss=0.6293 eval_acc=0.6617
epoch 6: train_loss=0.3953 train_acc=0.8075 eval_loss=0.6332 eval_acc=0.7114
epoch 7: train_loss=0.3437 train_acc=0.8363 eval_loss=0.6895 eval_acc=0.6716
epoch 8: train_loss=0.2806 train_acc=0.8850 eval_loss=0.6497 eval_acc=0.7214
epoch 9: train_loss=0.2236 train_acc=0.9100 eval_loss=0.7114 eval_acc=0.7164
epoch 10: train_loss=0.1584 train_acc=0.9387 eval_loss=1.0212 eval_acc=0.6368


In [16]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(50, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset.transform = train_transform_augmented

history_augmented = run_epochs(10)

with open("history_augmented.json", "w") as f:
    json.dump(history_augmented, f, indent=2)

epoch 1: train_loss=0.6444 train_acc=0.6400 eval_loss=0.6283 eval_acc=0.6866
epoch 2: train_loss=0.5865 train_acc=0.6787 eval_loss=0.7047 eval_acc=0.6766
epoch 3: train_loss=0.5940 train_acc=0.6613 eval_loss=0.5812 eval_acc=0.7114
epoch 4: train_loss=0.5469 train_acc=0.7150 eval_loss=0.6194 eval_acc=0.6816
epoch 5: train_loss=0.5469 train_acc=0.7113 eval_loss=0.6176 eval_acc=0.7065
epoch 6: train_loss=0.5207 train_acc=0.7275 eval_loss=0.5709 eval_acc=0.7065
epoch 7: train_loss=0.5528 train_acc=0.7037 eval_loss=0.5715 eval_acc=0.7114
epoch 8: train_loss=0.5313 train_acc=0.7250 eval_loss=0.6018 eval_acc=0.6965
epoch 9: train_loss=0.5075 train_acc=0.7450 eval_loss=0.6582 eval_acc=0.6517
epoch 10: train_loss=0.4897 train_acc=0.7638 eval_loss=0.5377 eval_acc=0.7313


In [17]:
with open("history_baseline.json") as f:
    baseline = json.load(f)
with open("history_augmented.json") as f:
    augmented = json.load(f)

print("Q3:", round(float(np.median(baseline["train_accuracy"])), 2))
print("Q4:", round(float(np.std(baseline["train_loss"], ddof=0)), 3))
print("Q5:", round(float(np.mean(augmented["evaluation_loss"])), 3))
print("Q6:", round(float(np.mean(augmented["evaluation_accuracy"][5:])), 2))

Q3: 0.81
Q4: 0.139
Q5: 0.609
Q6: 0.7
